# Import Libraries and Frameworks

In [2]:
#transforms are used for preprocessing Data.
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim

## Data Transformation Pipeline

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                        std=[0.229, 0.224, 0.225])
])

# Load Dataset

In [ ]:
full_dataset = datasets.ImageFolder('/Data', transform = transform)


print(f"total images:{len(full_dataset)}")
print("classes: " , full_dataset.classes)
print("class to idx: ", full_dataset.class_to_idx)



In [ ]:
#split the data into training, validation and testing data:
#split ratio:
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

train_size  = int(train_ratio*len(full_dataset))
val_size = int(val_ratio*len(full_dataset))
test_size = len(full_dataset)-train_size-val_size

#split the dataset:
train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


model = models.resnet18(pretrained = True)
model.fc = nn.Linear(model.fc.in_features, 2)


#------training our model phase, it was already pretrained on a bunch of classes.
#the loss function:
#it measures how far the prediction is from the correct answer
# prediction is bad -> high loss
# prediction is good -> low loss
criterion = nn.CrossEntropyLoss()

#Optimizer. updates the model's weights, it is an algorithm that decides how and by how much the model is updated
#the loss tells us how far is the prediction from the accurate answer
optimizer = optim.Adam(model.parameters(), lr = 0.0001)

# Model training

In [ ]:
for epoch in range(3):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()


    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    val_accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}, Train Loss: {total_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")




# Model Evaluation

In [11]:
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 98.64%


# Save the model

In [15]:
torch.save(model.state_dict(), "cat_dog_model.pth")
print("model saved successfully")

model saved successfully


# Make Predictions

In [17]:
from PIL import Image
import numpy as np

# Load pretrained ResNet18
model = models.resnet18(pretrained=True)

# Modify the final layer for binary classification (Cat/Dog)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)  # 2 classes: Cat, Dog

# Load your trained weights
model.load_state_dict(torch.load('cat_dog_model.pth'))
model.eval()  # Set to evaluation mode

print("Model loaded successfully!")

# 2. DEFINE THE SAME TRANSFORMS USED IN TRAINING
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet stats
                       std=[0.229, 0.224, 0.225])
])

# 3. PREDICTION FUNCTION
def predict_image(image_path, model, transform):
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0)  # Add batch dimension
    
    # Make prediction
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1)
        confidence = probabilities[0][predicted_class.item()].item()
        
        # Get all probabilities
        all_probs = probabilities[0].numpy()
    
    return predicted_class.item(), confidence, all_probs


# IMPORTANT: Make sure this matches your training
# Typically: index 0 = Cat, index 1 = Dog (or vice versa)
class_names = ['Cat', 'Dog']  # Index 0: Cat, Index 1: Dog

# 5. PREDICT ON YOUR IMAGE
# Path to image
image_path = "3.jpg"

# Make prediction
pred_idx, confidence, all_probs = predict_image(image_path, model, transform)

# Display results
print(f"Image: {image_path}")
print(f"Predicted Class: {class_names[pred_idx]}")
print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
print("All Probabilities:")
for i, class_name in enumerate(class_names):
    print(f"  {class_name}: {all_probs[i]:.4f} ({all_probs[i]*100:.2f}%)")


Model loaded successfully!
Image: 3.jpg
Predicted Class: Dog
Confidence: 1.0000 (100.00%)
All Probabilities:
  Cat: 0.0000 (0.00%)
  Dog: 1.0000 (100.00%)
